In [1]:
import logging
from typing import Any, ClassVar, Dict, List, Optional, Type

In [2]:
#TODO
import os
import sys
# Import pandas in a separate cell to avoid circular imports
import pandas as pd

In [3]:
#TODO Remove, just for notebook, hardcoding!
# %cd /Users/kunalagrawal/Desktop/Research/forked_ember/ember
# %pip install -e .

In [4]:
from ember.core.registry.operator.base.operator_base import Operator
from ember.core.registry.specification.specification import Specification
from ember.core.types.ember_model import EmberModel, Field

2025-03-22 14:17:46,380 [DEBUG] ConfigManager: Loading configuration...
2025-03-22 14:17:46,381 [DEBUG] ConfigManager: Configuration loaded successfully
2025-03-22 14:17:46,381 [INFO] ember.core.registry.model.initialization: Execute model discovery (timeout: 30 seconds per provider, running in parallel)
2025-03-22 14:17:46,912 [DEBUG] httpx: load_ssl_context verify=True cert=None trust_env=True http2=False
2025-03-22 14:17:46,913 [DEBUG] httpx: load_verify_locations cafile='/Users/kunalagrawal/anaconda3/lib/python3.11/site-packages/certifi/cacert.pem'
2025-03-22 14:17:46,935 [DEBUG] ember.core.registry.model.base.registry.discovery: OPENAI_API_KEY found, initialized OpenAIDiscovery successfully
2025-03-22 14:17:46,936 [INFO] ember.core.registry.model.base.registry.discovery: ANTHROPIC_API_KEY not found, skipping AnthropicDiscovery
2025-03-22 14:17:46,936 [INFO] ember.core.registry.model.base.registry.discovery: GOOGLE_API_KEY not found, skipping DeepmindDiscovery
2025-03-22 14:17:46,9

In [5]:
# 2) Import dataset loader/validator/sampler:
from ember.core.utils.data.base.loaders import HuggingFaceDatasetLoader, IDatasetLoader
from ember.core.utils.data.base.validators import IDatasetValidator, DatasetValidator
from ember.core.utils.data.base.samplers import IDatasetSampler, DatasetSampler
from ember.core.utils.data.base.models import DatasetInfo, DatasetEntry, TaskType
from ember.core.utils.data.base.preppers import IDatasetPrepper
from ember.core.utils.data.datasets_registry.mmlu import MMLUConfig
from ember.core.utils.data.datasets_registry.halueval import HaluEvalConfig

In [6]:
# 1) Import our dataset registry tools:
from ember.core.utils.data.registry import UnifiedDatasetRegistry, UNIFIED_REGISTRY, initialize_registry
from ember.core.utils.data.loader_factory import DatasetLoaderFactory
from ember.core.utils.data.initialization import initialize_dataset_registry

In [7]:
# 3) Import the DatasetService to use the pipeline:
from ember.core.utils.data.service import DatasetService

In [8]:
# 4) Import model and operator related modules
from ember.core import non
from ember.core.app_context import get_ember_context
from ember.core.registry.model.model_module.lm import LMModuleConfig, LMModule
from ember.core.registry.operator.base.operator_base import Operator
from ember.core.registry.specification.specification import Specification

In [9]:
# For settings
from ember.core.registry.model.config.settings import EmberSettings

In [10]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [11]:
# 1) Create a metadata registry and loader factory:
metadata_registry = UnifiedDatasetRegistry()
initialize_registry() #initialize with core datasets
loader_factory = DatasetLoaderFactory()

2025-03-22 14:17:50,072 [DEBUG] ember.core.utils.data.registry: Registered new dataset: truthful_qa
2025-03-22 14:17:50,073 [DEBUG] ember.core.utils.data.registry: Registered new dataset: mmlu
2025-03-22 14:17:50,073 [DEBUG] ember.core.utils.data.registry: Registered new dataset: commonsense_qa
2025-03-22 14:17:50,074 [DEBUG] ember.core.utils.data.registry: Registered new dataset: halueval
2025-03-22 14:17:50,074 [DEBUG] ember.core.utils.data.registry: Registered new dataset: my_shortanswer_ds
2025-03-22 14:17:50,074 [WARNING] ember.core.utils.data.registry: Could not import package: ember.data.datasets


In [12]:
# 3) Optionally, discover any additional plugin-based preppers:
loader_factory.discover_and_register_plugins()

2025-03-22 14:17:50,092 [INFO] ember.core.utils.data.loader_factory: Registered loader prepper for dataset: 'commonsense_qa'
2025-03-22 14:17:50,092 [INFO] ember.core.utils.data.loader_factory: Registered loader prepper for dataset: 'halueval'
2025-03-22 14:17:50,092 [INFO] ember.core.utils.data.loader_factory: Registered loader prepper for dataset: 'mmlu'
2025-03-22 14:17:50,093 [INFO] ember.core.utils.data.loader_factory: Registered loader prepper for dataset: 'short_answer'
2025-03-22 14:17:50,093 [INFO] ember.core.utils.data.loader_factory: Registered loader prepper for dataset: 'truthful_qa'
2025-03-22 14:17:50,093 [INFO] ember.core.utils.data.loader_factory: Auto-registered plugin preppers: ['commonsense_qa', 'halueval', 'mmlu', 'short_answer', 'truthful_qa']


In [13]:
# 4) Construct a dataset loader, validator, and sampler:
loader: IDatasetLoader = HuggingFaceDatasetLoader()
validator: IDatasetValidator = DatasetValidator()
sampler: IDatasetSampler = DatasetSampler()

In [14]:
# 5) Instantiate a DatasetService to handle load, validation, transform, sampling, and prep:
dataset_service = DatasetService(
    loader=loader,
    validator=validator,
    sampler=sampler,
    transformers=[]  # Insert any specialized transformers if needed
)

In [15]:
# Initialize the Ember context
# context = get_ember_context()

In [16]:
# Create LM modules for different models
g4o_config = LMModuleConfig(model_name="openai:gpt-4o", temperature=0.0)
g4o_mini_config = LMModuleConfig(model_name="openai:gpt-4o-mini", temperature=0.0)
g4o_mod = LMModule(config=g4o_config)
g4o_mini_mod = LMModule(config=g4o_mini_config)

2025-03-22 14:17:50,104 [DEBUG] ConfigManager: Loading configuration...
2025-03-22 14:17:50,104 [DEBUG] ConfigManager: Configuration loaded successfully
2025-03-22 14:17:50,105 [INFO] ember.core.registry.model.initialization: No models registered from configuration
2025-03-22 14:17:50,105 [INFO] ember.core.registry.model.initialization: Execute model discovery (timeout: 30 seconds per provider, running in parallel)
2025-03-22 14:17:50,106 [DEBUG] httpx: load_ssl_context verify=True cert=None trust_env=True http2=False
2025-03-22 14:17:50,106 [DEBUG] httpx: load_verify_locations cafile='/Users/kunalagrawal/anaconda3/lib/python3.11/site-packages/certifi/cacert.pem'
2025-03-22 14:17:50,126 [DEBUG] ember.core.registry.model.base.registry.discovery: OPENAI_API_KEY found, initialized OpenAIDiscovery successfully
2025-03-22 14:17:50,127 [INFO] ember.core.registry.model.base.registry.discovery: ANTHROPIC_API_KEY not found, skipping AnthropicDiscovery
2025-03-22 14:17:50,127 [INFO] ember.core.r

In [17]:
# # Create a hallucination detection operator
# # This is a simplified version - in a real implementation, you would use a specific
# # hallucination detection operator from the registry
# class HallucinationDetectionOperator(Operator):
#     """Operator that detects hallucinations in text."""
    
#     def __init__(self, lm_modules: List[LMModule]):
#         self.lm_modules = lm_modules
        
#     def __call__(self, *, query: str, choices: Dict[str, str]) -> Dict[str, Any]:
#         """Process the input to detect hallucinations.
        
#         Args:
#             query: The query text containing knowledge, question, and candidate answer
#             choices: The choices for hallucination detection (e.g., {"A": "Not Hallucinated", "B": "Hallucinated"})
            
#         Returns:
#             Dict with judgement and other metadata
#         """
#         judgements = []
        
#         # Format the prompt for hallucination detection
#         prompt = f"""You are evaluating whether a candidate answer is hallucinated or not.
        
# {query}

# Your task is to determine if the candidate answer is supported by the provided knowledge.
# If the answer is supported by the knowledge, respond with "A" (Not Hallucinated).
# If the answer is not supported by the knowledge, respond with "B" (Hallucinated).

# Respond with only "A" or "B".
# """
        
#         # Get judgements from each model
#         for lm_module in self.lm_modules:
#             response = lm_module(prompt=prompt)
#             # Extract just the letter from the response
#             for line in response.splitlines():
#                 line = line.strip()
#                 if line == "A" or line == "B":
#                     judgements.append(line)
#                     break
        
#         print("Judgements: ", judgements)
        
#         # Determine the final judgement (majority vote)
#         if judgements:
#             a_count = judgements.count("A")
#             b_count = judgements.count("B")
#             final_judgement = "A" if a_count > b_count else "B"
#         else:
#             final_judgement = "Unknown"
            
#         return {"judgement": final_judgement}

# # Set up the hallucination detection pipeline
# def setup_hallucination_detection():
#     # Create LM modules
#     g4o_mod = LMModule(config=g4o_config)
#     g4o_mini_mod = LMModule(config=g4o_mini_config)
    
#     # Create the operator with the LM modules
#     operator = HallucinationDetectionOperator(lm_modules=[g4o_mod, g4o_mini_mod])
    
#     # Create graph
#     graph_data = GraphData()
#     graph_data.add_node(
#         name="detector",
#         operator=operator,
#         inputs=[]
#     )
    
#     return graph_data

# # Function to run hallucination detection
# def run_hallucination_detection(**kwargs):
#     graph_data = setup_hallucination_detection()
    
#     input_data = {
#         "query": kwargs.get("query"),
#         "choices": kwargs.get("choices")
#     }
    
#     executor_service = GraphExecutorService()
#     results = executor_service.run(graph_data=graph_data, input_data=input_data)
    
#     if "detector" not in results:
#         return results
#     return results["detector"]

# # Example usage
# qa_example = {
#     "query": "Knowledge: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century. First for Women is a woman's magazine published by Bauer Media Group in the USA.\nQuestion: Which magazine was started first Arthur's Magazine or First for Women?\nCandidate Answer: First for Women was started first.. Is this candidate answer supported by the provided knowledge?",
#     "choices": {
#         "A": "Not Hallucinated",
#         "B": "Hallucinated"
#     },
#     "metadata": {
#         "correct_answer": "B"
#     }
# }

# # Run the example
# qa_result = run_hallucination_detection(**qa_example)
# print(f"\nQA Hallucination Check:")
# print(f"Query: {qa_example['query']}")
# print(f"Result: {qa_result}")

In [28]:
# Import NON components
from ember.core import non
from ember.core.registry.model.model_module.lm import LMModuleConfig, LMModule
from ember.core.registry.operator.base.operator_base import Operator
from ember.core.registry.specification.specification import Specification
from pydantic import BaseModel
from typing import Dict, Any, List, Optional

# Define input/output types
class HallucinationDetectionInputs(BaseModel):
    query: str
    choices: Dict[str, str]
    
class HallucinationDetectionOutputs(BaseModel):
    judgement: str

# Create the hallucination detection prompt template
class HallucinationDetectionSpecification(Specification):
    """Specification for hallucination detection."""
    
    def render_prompt(self, *, inputs: HallucinationDetectionInputs) -> str:
        """Renders the prompt for hallucination detection."""
        return f"""You are evaluating whether a candidate answer is hallucinated or not.
        
{inputs.query}

Your task is to determine if the candidate answer is supported by the provided knowledge.
If the answer is supported by the knowledge, respond with "A" (Not Hallucinated).
If the answer is not supported by the knowledge, respond with "B" (Hallucinated).

Respond with only "A" or "B".
"""

# Create a simple operator that formats the response
class FormatHallucinationResponse(Operator[Dict[str, Any], HallucinationDetectionOutputs]):
    """Formats the raw ensemble output into a clean hallucination detection result."""
    
    def forward(self, *, inputs: Dict[str, Any]) -> HallucinationDetectionOutputs:
        # Extract responses from the ensemble output
        responses = inputs.get("responses", [])
        
        # Clean up responses to get just A or B
        cleaned_responses = []
        for response in responses:
            for line in response.splitlines():
                line = line.strip()
                if line == "A" or line == "B":
                    cleaned_responses.append(line)
                    break
        
        print(f"Collected judgements: {cleaned_responses}")
        
        # Tally votes for A and B
        a_count = cleaned_responses.count("A")
        b_count = cleaned_responses.count("B")
        
        # Determine majority
        final_judgement = "A" if a_count > b_count else "B"
        
        return HallucinationDetectionOutputs(judgement=final_judgement)

# Setup function using NON components with different models
def setup_hallucination_detection():
    """Set up the hallucination detection pipeline using a varied ensemble of different models."""
    
    # Define different model configurations
    model_configs = [
        {"model_name": "openai:gpt-4o", "temperature": 0},
        {"model_name": "openai:gpt-4o-mini", "temperature": 0}, 
        {"model_name": "openai:gpt-3.5-turbo", "temperature": 0}
    ]
    
    # Create a varied ensemble with different models
    ensemble = non.VariedEnsemble(
        model_configs=model_configs)
    
    ensemble.specification.prompt_template="""
You are evaluating whether a candidate answer is hallucinated or not.

{query}

Your task is to determine if the candidate answer is supported by the provided knowledge.
If the answer is supported by the knowledge, respond with "A" (Not Hallucinated).
If the answer is not supported by the knowledge, respond with "B" (Hallucinated).

Respond with only "A" or "B".
"""
    
    # Format the responses to get the final judgement
    formatter = FormatHallucinationResponse()
    
    # Create a sequential pipeline: ensemble -> formatter
    pipeline = non.Sequential(operators=[ensemble, formatter])
    
    return pipeline

# Function to run hallucination detection
def run_hallucination_detection(**kwargs):
    """Run the hallucination detection pipeline."""
    
    # Set up the pipeline
    pipeline = setup_hallucination_detection()
    
    # Create input for the ensemble
    input_data = {
        "query": kwargs.get("query"),
        "choices": kwargs.get("choices", {"A": "Not Hallucinated", "B": "Hallucinated"})
    }
    
    # Run the pipeline
    result = pipeline(inputs=input_data)
    
    # Return result as a dictionary
    return {"judgement": result.judgement}

In [29]:
# Main Evaluation Logic
halu_df = pd.DataFrame(columns=['query','judgement','correct_answer'])

# Load HaluEval dataset
halu_info = UNIFIED_REGISTRY.get(name="halueval")
if not halu_info:
    raise ValueError("HaluEval dataset not properly registered.")

halu_prepper_class = loader_factory.get_prepper_class(dataset_name="halueval")
if not halu_prepper_class:
    raise ValueError("No HaluEval prepper found. Make sure it's registered.")

# Create config & prepper, defaulting to config_name="qa", split="data"
halu_config = HaluEvalConfig()
halu_prepper: IDatasetPrepper = halu_prepper_class(config=halu_config)

logger.info(f"Loading and preparing dataset: {halu_info.name}")
try:
    halu_dataset_entries: List[DatasetEntry] = dataset_service.load_and_prepare(
        dataset_info=halu_info.info,
        prepper=halu_prepper,
        config=halu_config,
        num_samples=3
    )
    logger.info(f"Received {len(halu_dataset_entries)} prepared entries for '{halu_info.name}'.")
    
    for i, entry in enumerate(halu_dataset_entries):
        data_entry = entry.model_dump()
        result = run_hallucination_detection(**data_entry)
        print(f"\nQA Hallucination:")
        print(f"[HaluEval] Entry #{i+1}:\n{data_entry}")
        print(f"Result: {result}")
        
        new_row = {
            "query": data_entry['query'], 
            "judgement": result["judgement"], 
            "correct_answer": data_entry['metadata']['correct_answer']
        } 
        halu_df = pd.concat([halu_df, pd.DataFrame([new_row])], ignore_index=True)
        
except Exception as e:
    logger.error(f"Error during HaluEval dataset preparation: {e}")

# Display results
halu_df

2025-03-22 14:51:01,512 [INFO] __main__: Loading and preparing dataset: halueval
2025-03-22 14:51:01,514 [INFO] ember.core.utils.data.service: [load_and_prepare] Starting process for dataset 'halueval' with dataset_name='pminervini/HaluEval', config='config_name='qa' split='data'', num_samples='3'.
2025-03-22 14:51:01,514 [INFO] ember.core.utils.data.service: [load_and_prepare] Converting configuration for loader compatibility.
2025-03-22 14:51:01,515 [INFO] ember.core.utils.data.service: [load_and_prepare] Resolved configuration: 'qa'.
2025-03-22 14:51:01,515 [INFO] ember.core.utils.data.service: [load_and_prepare] Loading data from dataset_name='pminervini/HaluEval'...
2025-03-22 14:51:01,516 [INFO] ember.core.utils.data.base.loaders: Checking dataset existence on the Hub: pminervini/HaluEval
2025-03-22 14:51:03,027 [DEBUG] urllib3.connectionpool: https://huggingface.co:443 "GET /api/datasets/pminervini/HaluEval HTTP/1.1" 200 3975
2025-03-22 14:51:03,031 [INFO] ember.core.utils.data.

,query,judgement,correct_answer
